In [204]:
#  Import libraries
import numpy as np
import pandas as pd
import re
import requests
import string
import time
import random
import datetime
from bs4 import BeautifulSoup
from rapidfuzz import process, fuzz

In [185]:
#  Import dataset
df_top500 = pd.read_excel("data/top500.xlsx")
df_top500.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   NR                     500 non-null    int64  
 1   Bedrijfsnaam           500 non-null    object 
 2   Activiteit             500 non-null    object 
 3   Soort bedrijf          500 non-null    object 
 4   Omzet (in mln)         500 non-null    int64  
 5   Winst (in mln)         500 non-null    object 
 6   Aantal mw              500 non-null    object 
 7   ICT Functies (Ja/Nee)  0 non-null      float64
 8   Functies/Zoektermen    0 non-null      float64
 9   Bronvermeldingen       0 non-null      float64
 10  Contact                33 non-null     object 
dtypes: float64(3), int64(2), object(6)
memory usage: 43.1+ KB


In [186]:
df_top500.head()

,NR,Bedrijfsnaam,Activiteit,Soort bedrijf,Omzet (in mln),Winst (in mln),Aantal mw,ICT Functies (Ja/Nee),Functies/Zoektermen,Bronvermeldingen,Contact
0,1,Vitol Holding,Oliehandel,Personeel,370000,-,1700,NaN,NaN,NaN,NaN
1,2,Ahold Delhaize,Supermarktketen,Beurs,88649,1874,402000,NaN,NaN,NaN,NaN
2,3,ING,Bank,Beurs,60189,7521,59434,NaN,NaN,NaN,Ja
3,4,Louis Dreyfus,Grondstoffenhandelaar,Familie,46777,933,18426,NaN,NaN,NaN,NaN
4,5,Ingka Holding (IKEA),Meubelwarenhuisketen,Familie,44300,1507,165353,NaN,NaN,NaN,NaN


In [187]:
# Dropping irrelevant columns
df_top500.drop(columns=["Activiteit", "Soort bedrijf", "Omzet (in mln)", "Winst (in mln)", "Aantal mw"], inplace=True)

# Renaming columns 
df_top500.rename(columns={
    "NR": "id",
    "Bedrijfsnaam": "company_name",
    "ICT Functies (Ja/Nee)":"has_openings",
    "Functies/Zoektermen": "search_results",
    "Bronvermeldingen": "source",
    "Contact": "contact"
}, 
inplace=True)

# Adding column for company page URL
df_top500["url"] = np.nan

df_top500.head()

,id,company_name,has_openings,search_results,source,contact,url
0,1,Vitol Holding,NaN,NaN,NaN,NaN,NaN
1,2,Ahold Delhaize,NaN,NaN,NaN,NaN,NaN
2,3,ING,NaN,NaN,NaN,Ja,NaN
3,4,Louis Dreyfus,NaN,NaN,NaN,NaN,NaN
4,5,Ingka Holding (IKEA),NaN,NaN,NaN,NaN,NaN


In [188]:
#  Filtering out companies where contact is established
df_top500 = df_top500.loc[df_top500["contact"].isna()]
df_top500.drop(columns=["contact"], inplace=True)
df_top500.info()

<class 'pandas.core.frame.DataFrame'>
Index: 467 entries, 0 to 499
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              467 non-null    int64  
 1   company_name    467 non-null    object 
 2   has_openings    0 non-null      float64
 3   search_results  0 non-null      float64
 4   source          0 non-null      float64
 5   url             0 non-null      float64
dtypes: float64(4), int64(1), object(1)
memory usage: 25.5+ KB


In [189]:
#  Cleaning company names (removing parentheses and anything between them) and sorting them alphabetically
df_top500["company_name"] = df_top500["company_name"].apply(lambda x: re.sub(r"\(.*?\)", "", x).strip())
df_top500.sort_values(by="company_name", inplace=True)
df_top500.head(30)

,id,company_name,has_openings,search_results,source,url
242,243,A fen,NaN,NaN,NaN,NaN
387,388,A. Hak,NaN,NaN,NaN,NaN
292,293,ABZ Diervoeding,NaN,NaN,NaN,NaN
46,47,ACT Commodities,NaN,NaN,NaN,NaN
204,205,ADG Dienstengroep,NaN,NaN,NaN,NaN
359,360,AFAS,NaN,NaN,NaN,NaN
491,492,AGAR Holding,NaN,NaN,NaN,NaN
149,150,APG Groep,NaN,NaN,NaN,NaN
67,68,ASM,NaN,NaN,NaN,NaN
140,141,ASVB,NaN,NaN,NaN,NaN


In [190]:
def scrape_company_names():    
    # Scraping all companies in ICTergezocht client base
    BASE_URL = "https://www.ictergezocht.nl/ict-bedrijven/bedrijf-start-met-{}/"

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
    }

    ictergezocht_data = pd.DataFrame(columns=["company_name", "company_url"])

    # Loop through the alphabet and scrape each page
    for letter in string.ascii_lowercase:
        url = BASE_URL.format(letter)

        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            print(f"Successfully retrieved page: {url}")
        except requests.RequestException as e:
            print(f"Failed to retrieve page {url}: {e}")
            continue

        soup = BeautifulSoup(response.text, 'html.parser')

        company_links = soup.select('a[href^="https://www.ictergezocht.nl/ict-bedrijven/"]')

        for link in company_links:
            company_name = link.get('title')
            company_url = link['href']

            if company_name and company_url:
                company_name = company_name.replace("Bedrijven ", "").strip()
                ictergezocht_data.loc[len(ictergezocht_data)] = {"company_name": company_name, "company_url": company_url}
        
        time.sleep(random.uniform(1, 3))

        return ictergezocht_data

# ictergezocht_data = scrape_company_names()
# This only needed to be run once
# Saving scraped data to csv so we don't have to continously scrape the website
# ictergezocht_data.to_csv("data/ictergezocht_companies.csv")

In [191]:
# After initially scraping the website we can load from csv
ictergezocht_data = pd.read_csv("data/ictergezocht_companies.csv", )
ictergezocht_data.drop(columns=["Unnamed: 0"], inplace=True)

In [192]:
ictergezocht_data.head()

,company_name,company_url
0,ICT bedrijven zoeken,https://www.ictergezocht.nl/ict-bedrijven/
1,ICT bedrijven zoeken,https://www.ictergezocht.nl/ict-bedrijven/
2,a friend of mine,https://www.ictergezocht.nl/ict-bedrijven/4763...
3,A New Spring,https://www.ictergezocht.nl/ict-bedrijven/703-...
4,A!tention,https://www.ictergezocht.nl/ict-bedrijven/64-a...


In [193]:
#  Searching for exact matches in company 500

df_top500["company_name_lower"] = df_top500["company_name"].str.lower()
ictergezocht_data["company_name_lower"] = ictergezocht_data["company_name"].str.lower()

for company, url in zip(ictergezocht_data["company_name_lower"], ictergezocht_data["company_url"]):
    if company in df_top500["company_name_lower"].values:
        df_top500.loc[df_top500["company_name_lower"] == company, "source"] = "ictergezocht"
        df_top500.loc[df_top500["company_name_lower"] == company, "url"] = url


/tmp/ipykernel_9898/3714999572.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'ictergezocht' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_top500.loc[df_top500["company_name_lower"] == company, "source"] = "ictergezocht"
/tmp/ipykernel_9898/3714999572.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'https://www.ictergezocht.nl/ict-bedrijven/10678-agrico/' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_top500.loc[df_top500["company_name_lower"] == company, "url"] = url


In [194]:
df_top500.info()
# Notice 51 exact matches are found

<class 'pandas.core.frame.DataFrame'>
Index: 467 entries, 242 to 367
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  467 non-null    int64  
 1   company_name        467 non-null    object 
 2   has_openings        0 non-null      float64
 3   search_results      0 non-null      float64
 4   source              51 non-null     object 
 5   url                 51 non-null     object 
 6   company_name_lower  467 non-null    object 
dtypes: float64(2), int64(1), object(4)
memory usage: 29.2+ KB


In [195]:
df_top500.loc[~df_top500["source"].isna()]

,id,company_name,has_openings,search_results,source,url,company_name_lower
67,68,ASM,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/782-...,asm
281,282,Agrico,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1067...,agrico
409,410,Amac,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/9843...,amac
332,333,Autobedrijf Van den Udenhout,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/9874...,autobedrijf van den udenhout
340,341,Bakker Logistiek,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1019...,bakker logistiek
145,146,Basic-Fit,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/5107...,basic-fit
325,326,Batenburg Techniek,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1035...,batenburg techniek
296,297,Bejo Zaden,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1045...,bejo zaden
405,406,Brisker Groep,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1089...,brisker groep
460,461,CRV,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1016...,crv


In [196]:
#  Search for company names that are very similar for manual review using fuzzy matching

matches = []
scores = []
threshold = 90

for name in ictergezocht_data["company_name_lower"]:
    best_match, score, _ = process.extractOne(name, df_top500["company_name_lower"], scorer=fuzz.ratio)
    if score == 100:
        matches.append(None)
        scores.append(score)
        continue
    matches.append(best_match if score > threshold else None)
    scores.append(score)

ictergezocht_data["matched_name"] = matches
ictergezocht_data["match_score"] = scores
ictergezocht_data["manual_review"] = (ictergezocht_data["match_score"] > threshold) & (ictergezocht_data["match_score"] < 100)

df_top500.drop(columns=["company_name_lower"], inplace=True)
ictergezocht_data.drop(columns=["company_name_lower"], inplace=True)

ictergezocht_data["manual_review"].value_counts()


manual_review
False    4825
True       10
Name: count, dtype: int64

In [197]:
ictergezocht_data[ictergezocht_data["manual_review"]]

,company_name,company_url,matched_name,match_score,manual_review
220,AR Holding,https://www.ictergezocht.nl/ict-bedrijven/1068...,agar holding,90.909091,True
690,CC Group,https://www.ictergezocht.nl/ict-bedrijven/4856...,ccv group,94.117647,True
980,Data Professionals,https://www.ictergezocht.nl/ict-bedrijven/1050...,atlas professionals,91.891892,True
1994,Infotheek Groep,https://www.ictergezocht.nl/ict-bedrijven/1533...,infotheek group,93.333333,True
2488,Maandag®,https://www.ictergezocht.nl/ict-bedrijven/9988...,maandag,93.333333,True
2489,Maandag®,https://www.ictergezocht.nl/ict-bedrijven/530-...,maandag,93.333333,True
2490,Maandag®,https://www.ictergezocht.nl/ict-bedrijven/7983...,maandag,93.333333,True
2491,Maandag®,https://www.ictergezocht.nl/ict-bedrijven/321-...,maandag,93.333333,True
2687,Monta,https://www.ictergezocht.nl/ict-bedrijven/9263...,monuta,90.909091,True
4119,Tembogroup,https://www.ictergezocht.nl/ict-bedrijven/8817...,tembo group,95.238095,True


In [198]:
df_top500.loc[df_top500["company_name"] == "Infotheek Group", "url"] = ictergezocht_data.loc[ictergezocht_data["company_name"] == "Infotheek Groep", "company_url"].iloc[0]
df_top500.loc[df_top500["company_name"] == "Infotheek Group", "source"] = "ictergezocht"

df_top500.loc[df_top500["company_name"] == "Maandag", "url"] = ictergezocht_data.loc[ictergezocht_data["company_name"] == "Maandag®", "company_url"].iloc[0]
df_top500.loc[df_top500["company_name"] == "Maandag", "source"] = "ictergezocht"

df_top500.loc[df_top500["company_name"] == "Tembo Group", "url"] = ictergezocht_data.loc[ictergezocht_data["company_name"] == "Tembogroup", "company_url"].iloc[0]
df_top500.loc[df_top500["company_name"] == "Tembo Group", "source"] = "ictergezocht"


In [199]:
df_top500[~df_top500["source"].isna()]

,id,company_name,has_openings,search_results,source,url
67,68,ASM,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/782-...
281,282,Agrico,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1067...
409,410,Amac,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/9843...
332,333,Autobedrijf Van den Udenhout,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/9874...
340,341,Bakker Logistiek,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1019...
145,146,Basic-Fit,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/5107...
325,326,Batenburg Techniek,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1035...
296,297,Bejo Zaden,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1045...
405,406,Brisker Groep,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1089...
460,461,CRV,NaN,NaN,ictergezocht,https://www.ictergezocht.nl/ict-bedrijven/1016...


In [200]:
# Removing url prefix from entries

URL_PREFIX_ICTERGEZOCHT = "https://www.ictergezocht.nl/ict-bedrijven/"
df_top500["url"] = df_top500["url"].str.replace(URL_PREFIX_ICTERGEZOCHT, "")
df_ictergezocht = df_top500[~df_top500["source"].isna()]

df_ictergezocht

,id,company_name,has_openings,search_results,source,url
67,68,ASM,NaN,NaN,ictergezocht,782-asm/
281,282,Agrico,NaN,NaN,ictergezocht,10678-agrico/
409,410,Amac,NaN,NaN,ictergezocht,9843-amac/
332,333,Autobedrijf Van den Udenhout,NaN,NaN,ictergezocht,9874-autobedrijf-van-den-udenhout/
340,341,Bakker Logistiek,NaN,NaN,ictergezocht,10197-bakker-logistiek/
145,146,Basic-Fit,NaN,NaN,ictergezocht,5107-basic-fit/
325,326,Batenburg Techniek,NaN,NaN,ictergezocht,10353-batenburg-techniek/
296,297,Bejo Zaden,NaN,NaN,ictergezocht,10457-bejo-zaden/
405,406,Brisker Groep,NaN,NaN,ictergezocht,10895-brisker-groep/
460,461,CRV,NaN,NaN,ictergezocht,10161-crv/


In [201]:
# Scraper for all jobpostings

def scrape_jobpostings_ictergezocht(data, BASE_URL):
    job_postings_ictergezocht = pd.DataFrame(columns=["company", "title", "url"])
    
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"
        }
    
    for i, row in data.iterrows():
        company_url = row["url"]
        url = BASE_URL + company_url

        try:
            response = requests.get(url, headers=headers)
            response.raise_for_status()
            print(f"Successfully retrieved page: {url}")
        except requests.RequestException as e:
            print(f"Failed to retrieve page {url}: {e}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")
        open_vacs_element = soup.find(id="open_vacs")

        if open_vacs_element:
            data.at[i, "has_openings"] = 1
            jobpostings = open_vacs_element.find_all("h3")

            for job in jobpostings:
                a = job.find("a")
                job_title = a["title"]
                job_url = a["href"]

                new = pd.DataFrame({
                    "company": [row["company_name"]],
                    "title": [job_title],
                    "url": [job_url]
                })

                job_postings_ictergezocht = pd.concat([job_postings_ictergezocht, new], ignore_index=True)
        else:
            data.at[i, "has_openings"] = 0
        
        time.sleep(random.uniform(1, 3))

    return job_postings_ictergezocht

job_postings_ictergezocht = scrape_jobpostings_ictergezocht(df_ictergezocht, URL_PREFIX_ICTERGEZOCHT)

Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/782-asm/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/10678-agrico/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/9843-amac/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/9874-autobedrijf-van-den-udenhout/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/10197-bakker-logistiek/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/5107-basic-fit/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/10353-batenburg-techniek/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/10457-bejo-zaden/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/10895-brisker-groep/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/10161-crv/
Successfully retrieved page: https://www.ictergezocht.nl/ict-bedrijven/8841-chipsoft/
Successfully retrie

In [202]:
job_postings_ictergezocht

,company,title,url
0,Brisker Groep,ICT Medewerker,https://www.ictergezocht.nl/ict-vacatures-in/a...


In [203]:
date = datetime.date.today()
job_postings_ictergezocht.to_excel(f"data/results_{date}.xlsx")